# COMP 3610 Assignment 4: MLOps & Model Deployment

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn

import joblib

import json
import os

## Part 1: Experiment Tracking with MLflow

### Task 1.1: MLflow Setup & Experiment Logging

The MLflow tracking URI is set to the locally running MLflow server, and a new experiment called `taxi-tip-prediction` is created.

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("taxi-tip-prediction")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1776283271098, experiment_id='1', last_update_time=1776283271098, lifecycle_stage='active', name='taxi-tip-prediction', tags={}, trace_location=None, workspace='default'>

#### Dataset Loading, Cleaning and Preprocessing

In accordance with the assignment instructions, the taxi dataset is recreated programmatically from the raw NYC Yellow Taxi January 2024 parquet file and the taxi zone lookup file before continuing with feature engineering and model training. The same regression preprocessing steps are applied from assignment 2 so that the deployed and tracked models remain consistent with the earlier assignment.

In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Programmatically download the datasets
import requests
import os

os.makedirs("data/raw", exist_ok=True)

tripdata_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

def download_file(url, save_path):
    try:
        r = requests.get(url)
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            f.write(r.content)
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")

download_file(tripdata_url, "data/raw/yellow_tripdata_2024-01.parquet")
download_file(zone_url, "data/raw/taxi_zone_lookup.csv")

In [4]:
def recreate_cleaned_taxi_dataset(raw_trip_path: Path, zone_lookup_path: Path) -> pd.DataFrame:
    df = pd.read_parquet(raw_trip_path).copy()
    zones = pd.read_csv(zone_lookup_path).copy()

    critical_cols = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "fare_amount"
    ]
    df = df.dropna(subset=critical_cols)

    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

    df = df[(df["trip_distance"] > 0) & (df["trip_distance"] <= 100)]
    df = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 500)]
    df = df[df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]]

    df["trip_duration_minutes"] = (
        (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    )
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.strftime("%A")

    df["trip_speed_mph"] = np.where(
        df["trip_duration_minutes"] > 0,
        df["trip_distance"] / (df["trip_duration_minutes"] / 60),
        0
    )

    # Join taxi zone lookup twice
    pickup_zones = zones[["LocationID", "Zone", "Borough"]].rename(columns={
        "LocationID": "PULocationID",
        "Zone": "pickup_zone",
        "Borough": "pickup_borough"
    })

    dropoff_zones = zones[["LocationID", "Zone", "Borough"]].rename(columns={
        "LocationID": "DOLocationID",
        "Zone": "dropoff_zone",
        "Borough": "dropoff_borough"
    })

    df = df.merge(pickup_zones, on="PULocationID", how="left")
    df = df.merge(dropoff_zones, on="DOLocationID", how="left")

    return df

df = recreate_cleaned_taxi_dataset("data/raw/yellow_tripdata_2024-01.parquet", "data/raw/taxi_zone_lookup.csv")
os.makedirs("data/processed", exist_ok=True)
df.to_parquet("data/processed/cleaned_taxi.parquet", index=False)
print(f"Saved cleaned dataset to data/processed/cleaned_taxi.parquet")

print(df.shape)
df.head()

Saved cleaned dataset to data/processed/cleaned_taxi.parquet
(2869527, 27)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,congestion_surcharge,Airport_fee,trip_duration_minutes,pickup_hour,pickup_day_of_week,trip_speed_mph,pickup_zone,pickup_borough,dropoff_zone,dropoff_borough
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,...,2.5,0.0,19.800000,0,Monday,5.212121,Penn Station/Madison Sq West,Manhattan,East Village,Manhattan
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,...,2.5,0.0,6.600000,0,Monday,16.363636,Lenox Hill East,Manhattan,Upper East Side North,Manhattan
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,...,2.5,0.0,17.916667,0,Monday,15.739535,Upper East Side North,Manhattan,East Village,Manhattan
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,...,2.5,0.0,8.300000,0,Monday,10.120482,East Village,Manhattan,SoHo,Manhattan
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,...,2.5,0.0,6.100000,0,Monday,7.868852,SoHo,Manhattan,Lower East Side,Manhattan


In [5]:
PATH = "data/processed/cleaned_taxi.parquet"
df = pd.read_parquet(PATH)
df = df[df["payment_type"] == 1].copy()

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek
df["is_weekend"] = (df["pickup_day_of_week"] >= 5).astype(int)

df["log_trip_distance"] = np.log1p(df["trip_distance"])
df["fare_per_mile"] = df["fare_amount"] / df["trip_distance"].replace(0, 1)
df["fare_per_minute"] = df["fare_amount"] / df["trip_duration_minutes"].replace(0, 1)

df = pd.get_dummies(df, columns=["pickup_borough", "dropoff_borough"])
df = df.astype({col: "int" for col in df.columns if "borough" in col})

df = df.drop(columns=[
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_zone",
    "dropoff_zone",
    "total_amount",
    "store_and_fwd_flag"
])

X = df.drop(columns=["tip_amount"])
y_reg = df["tip_amount"]

feature_columns = list(X.columns)

X_train, X_temp, y_train_reg, y_temp_reg = train_test_split(
    X, y_reg, test_size=0.3, random_state=42
)
X_val, X_test, y_val_reg, y_test_reg = train_test_split(
    X_temp, y_temp_reg, test_size=0.5, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Training samples: ", len(X_train))
print("Validation samples: ", len(X_val))
print("Testing samples: ", len(X_test))

Training samples:  1608830
Validation samples:  344749
Testing samples:  344750


In [6]:
# Save feature columns and scaler for later use in prediction
os.makedirs("models", exist_ok=True)

with open("models/feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

The first regression model logged is a Linear Regression baseline. MLflow records the model type, evaluation metrics, and trained model artifact.

In [7]:
with mlflow.start_run(run_name="linear-regression"):

    lr = LinearRegression()
    lr.fit(X_train, y_train_reg)

    preds = lr.predict(X_test)

    mae = mean_absolute_error(y_test_reg, preds)
    rmse = np.sqrt(mean_squared_error(y_test_reg, preds))
    r2 = r2_score(y_test_reg, preds)

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.set_tag("model_type", "LinearRegression")
    mlflow.set_tag("dataset_version", "assignment2_v1")

    mlflow.sklearn.log_model(lr, name="model")

    print({
        "model": "LinearRegression",
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    })

2026/04/16 11:34:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model': 'LinearRegression', 'mae': 1.2122024735979027, 'rmse': np.float64(2.382156376231309), 'r2': 0.6229095591625703}
🏃 View run linear-regression at: http://127.0.0.1:5001/#/experiments/1/runs/ab8c00dedbe843d19d6a1cfe94b32cdd
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


The second regression model logged is a Random Forest Regressor. Its configuration, evaluation metrics, and artifact are tracked in MLflow for later comparison.

In [8]:
with mlflow.start_run(run_name="random-forest-regressor"):

    rf = RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train_reg)

    preds = rf.predict(X_test)

    mae = mean_absolute_error(y_test_reg, preds)
    rmse = np.sqrt(mean_squared_error(y_test_reg, preds))
    r2 = r2_score(y_test_reg, preds)

    mlflow.log_params({
        "model_type": "RandomForestRegressor",
        "n_estimators": 50,
        "max_depth": 20,
        "random_state": 42,
        "n_jobs": -1
    })
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.set_tag("model_type", "RandomForestRegressor")
    mlflow.set_tag("dataset_version", "assignment2_v1")

    mlflow.sklearn.log_model(rf, name="model")

    print({
        "model": "RandomForestRegressor",
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    })

2026/04/16 11:37:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model': 'RandomForestRegressor', 'mae': 1.1899463200978007, 'rmse': np.float64(2.3827898196579813), 'r2': 0.6227089869230966}
🏃 View run random-forest-regressor at: http://127.0.0.1:5001/#/experiments/1/runs/1365bd959c884494883aa5d8185b2339
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


After both runs are logged, the MLflow UI is opened to verify that the experiment contains both models, their parameters, metrics, tags, and saved artifacts.

![Figure 1](images/figure1.png)

*Figure 1 shows the the MLflow UI which provides a central view of the experiment for comparison in Task 1.2*

Both regression models were successfully logged to the `taxi-tip-prediction` experiment in MLflow. For each run, the parameters, evaluation metrics (MAE, RMSE, and R²), tags, and trained model artifact were recorded. 

### Task 1.2: Model Comparison & Registry 

In this section, the logged regression models are compared using MLflow. The better-performing model is then registered in the MLflow Model Registry, documented with a version description, and loaded back for a sample prediction.

In [9]:
experiment = mlflow.get_experiment_by_name("taxi-tip-prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

runs[[
    "run_id",
    "tags.mlflow.runName",
    "metrics.mae",
    "metrics.rmse",
    "metrics.r2"
]]

,run_id,tags.mlflow.runName,metrics.mae,metrics.rmse,metrics.r2
0,1365bd959c884494883aa5d8185b2339,random-forest-regressor,1.189946,2.382790,0.622709
1,ab8c00dedbe843d19d6a1cfe94b32cdd,linear-regression,1.212202,2.382156,0.622910


![Figure 2](images/figure2.png)

![Figure 3](images/figure3.png)

*Figures 2 and 3 show the MLflow comparison view for the Linear Regression and Random Forest Regressor runs.*

The Random Forest Regressor slightly outperformed the Linear Regression model, achieving a lower MAE (1.19 vs 1.212), which indicates better prediction accuracy. Both models produced the same RMSE and R² values, suggesting similar overall fit. Based on the lower MAE, the Random Forest Regressor was selected as the better-performing model.

The Random Forest likely performed better due to its ability to capture non-linear relationships between features such as trip distance and fare amount, which linear regression cannot model effectively.

In [10]:
# Saving the best model (Random Forest Regressor) locally using joblib
joblib.dump(rf, "models/taxi_tip_model.pkl")

# Saving model metadata
model_metadata = {
    "model_name": "taxi-tip-regressor",
    "version": "1.0.0",
    "features": feature_columns,
    "metrics": {
        "mae": 1.19,
        "rmse": 2.382,
        "r2": 0.623
    }
}

with open("models/model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

In [11]:
best_run = runs.loc[runs["metrics.mae"].idxmin()]
best_run_id = best_run["run_id"]

best_run_id

'1365bd959c884494883aa5d8185b2339'

In [12]:
# Register the best model to the MLflow Model Registry
result = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="taxi-tip-regressor"
)

print(result)

Registered model 'taxi-tip-regressor' already exists. Creating a new version of this model...
2026/04/16 11:37:49 WARNING mlflow.tracking._model_registry.fluent: Run with id 1365bd959c884494883aa5d8185b2339 has no artifacts at artifact path 'model', registering model based on models:/m-0147fe4f56eb4923854272feb6a3a9d7 instead
2026/04/16 11:37:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor, version 3


<ModelVersion: aliases=[], creation_timestamp=1776353869896, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1776353869896, metrics=None, model_id=None, name='taxi-tip-regressor', params=None, run_id='1365bd959c884494883aa5d8185b2339', run_link='', source='models:/m-0147fe4f56eb4923854272feb6a3a9d7', status='READY', status_message=None, tags={}, user_id='', version='3', workspace='default'>


Created version '3' of model 'taxi-tip-regressor'.


In [13]:
# Add a version description to the registered model in MLflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.update_model_version(
    name="taxi-tip-regressor",
    version=result.version,
    description="Random Forest Regressor selected from Assignment 4 comparison. MAE=1.19, RMSE=2.382, R2=0.623. Trained on Assignment 2 cleaned taxi data."
)

<ModelVersion: aliases=[], creation_timestamp=1776353869896, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description=('Random Forest Regressor selected from Assignment 4 comparison. MAE=1.19, '
 'RMSE=2.382, R2=0.623. Trained on Assignment 2 cleaned taxi data.'), last_updated_timestamp=1776353869957, metrics=None, model_id=None, name='taxi-tip-regressor', params=None, run_id='1365bd959c884494883aa5d8185b2339', run_link='', source='models:/m-0147fe4f56eb4923854272feb6a3a9d7', status='READY', status_message=None, tags={}, user_id='', version='3', workspace='default'>

![Figure 4](images/figure4.png)

*Figure 4 shows the registered model `taxi-tip-regressor` in the MLflow Model Registry. The selected model (Random Forest Regressor) is stored as Version 1 along with its performance metrics and description.*

The registered model is loaded back from the MLflow Model Registry to verify that it can be retrieved successfully for deployment use.

In [14]:
registered_model = mlflow.sklearn.load_model(f"models:/taxi-tip-regressor/{result.version}")

print("Model loaded successfully from registry.")

Model loaded successfully from registry.


A sample prediction is generated using the registered model to confirm that the model can be loaded and used for inference successfully.

In [15]:
sample_input = X_test[:1]
sample_prediction = registered_model.predict(sample_input)

print("Sample prediction:", sample_prediction[0])

Sample prediction: 2.3548894686124173


The logged regression models were compared using both the MLflow UI and the MLflow API. Based on the comparison, the Random Forest Regressor was selected as the better-performing model and registered in the MLflow Model Registry as `taxi-tip-regressor`. The model was then successfully loaded from the registry and used to generate a sample prediction.

## Part 2: Model Serving with FastAPI

### Task 2.1: API Design & Implementation 

In this section, a FastAPI application is created to serve predictions from the selected regression model in `app.py` . The model and preprocessing artifacts are loaded once at application startup, and a `/predict` endpoint is implemented with validated input fields and a structured JSON response.

![Figure 5](images/figure5.png)

*Figure 5 shows the FastAPI Swagger UI interface with the `/predict` endpoint.*

![Figure 6](images/figure6.png)

*Figure 6 shows a sample JSON request used to test the `/predict` endpoint.*

![Figure 7](images/figure7.png)

*Figure 7 shows a successful response from the `/predict` endpoint. The API returns the predicted tip amount along with a unique prediction identifier and model version.*



The FastAPI application loads the trained model, scaler, feature column definitions, and metadata once at startup using a lifespan handler. The `/predict` endpoint accepts validated trip features, applies the same preprocessing used during training, and returns the predicted tip amount along with the model version and a unique prediction identifier.

### Task 2.2: Enhanced API Features

![Figure 8](images/figure8.png)

*Figure 8 shows all implemented API endpoints including batch prediction, health check, and model information.*

![Figure 9](images/figure9.png)

![Figure 10](images/figure10.png)

*Figures 9 and 10 shows a batch prediction request and response.*
 The API accepts multiple records and returns predictions along with processing time.

![Figure 11](images/figure11.png)

*Figure 11 shows the health check endpoint confirming that the API is running and the model is loaded.*


![Figure 12](images/figure12.png)

*Figure 12 shows the model metadata including features and version.*

The API was extended with additional functionality to support batch prediction, system monitoring, and model transparency. The `/predict/batch` endpoint allows up to 100 records to be processed in a single request, improving efficiency for bulk predictions. The `/health` endpoint provides real-time information about API status and model availability. The `/model/info` endpoint exposes metadata such as model version, features, and evaluation metrics. A global exception handler was also implemented to ensure consistent and secure error responses.

### Task 2.3: API Testing 

![Figure 13](images/figure13.png)

*Figure 13 shows the Swagger UI documentation for the API, confirming that all endpoints are available for testing.*

![Figure 14](images/figure14.png)

*Figure 14 shows the pytest results for the FastAPI application. All tests passed successfully.*


Automated tests were implemented using `pytest` and FastAPI’s `TestClient` to validate API functionality. The test suite covered successful single and batch predictions, invalid input handling, health checks, model information retrieval, and edge cases. All tests passed successfully, confirming that the API behaves as expected.

## Part 3: Containerization with Docker

### Task 3.1: Dockerfile & Image Building

The FastAPI prediction service was containerized using Docker. The image was built successfully from the Dockerfile and had a size of approximately 1.57GB. The container was then executed with port mapping, allowing the API to be accessed externally. The `/health` endpoint confirmed that the service was running correctly, and the `/predict` endpoint returned a valid prediction, demonstrating that the model was successfully loaded and operational within the container.

![Figure 15](images/figure15.png)

*Figure 15 shows the Docker image being successfully built from the Dockerfile.*

![Figure 16](images/figure16.png)

*Figure 16 shows the created Docker image and its size as reported by the `docker images` command.*

![Figure 17](images/figure17.png)

*Figure 17 shows the containerized API responding successfully to a health check request.*

![Figure 18](images/figure18.png)

*Figure 18 shows the API returning a valid prediction response from within the Docker container.*

### Task 3.2: Docker Compose & Deployment Demo

*docker compose up*

![Figure 18](images/figure18_docker_compose_up.png)

The services were started using `docker compose up`, which initializes both the API and MLflow containers. As shown below, both services were successfully created and started, with the API running on port 8000 (mapped to 8001 externally) and MLflow running on port 5000 (mapped to 5001 externally).

*docker compose ps*

![Figure 19](images/figure19_docker_compose_ps.png)

The status of the running services was verified using `docker compose ps`. This confirms that both the API and MLflow containers are active and correctly mapped to their respective ports, demonstrating that the multi-container setup is functioning as expected.

*prediction requests*

![Figure 20](images/figure20_prediction_requests.png)

The deployed API was tested by sending multiple prediction requests using `curl`. Three separate requests were made with different input values, and all returned valid predictions. This confirms that the API is fully functional within the Docker Compose environment.

*docker compose down*

![Figure 21](images/figure21_docker_compose_down.png)

After testing, the services were stopped using `docker compose down`, which cleanly shuts down and removes the containers and associated network. This demonstrates proper lifecycle management of the deployment.

*Bonus* 

![Figure 22](images/figure22_bonus.png)

To verify inter-container communication, a request was made from within the API container to the MLflow service using Python’s `requests` library. The request to `http://mlflow:5000` returned a `200` status code, confirming that the API container can successfully communicate with the MLflow container using Docker Compose service names.

This demonstrates that the Docker Compose networking is correctly configured, allowing services to interact with each other internally without relying on external ports.

#### Container Size

The Docker image for the FastAPI prediction service was successfully built with a size of approximately 1.57GB, as shown using the `docker images` command. The size is primarily due to the inclusion of machine learning libraries such as pandas, NumPy, and scikit-learn required for preprocessing and prediction.

#### Setup Instructions

To run the application locally using Docker Compose:

1. Build and start the services:
docker compose up


2. Access the API at:
http://localhost:8001


3. Access the MLflow UI (bonus service):
http://localhost:5001


4. Send prediction requests using:
curl -X POST http://localhost:8001/predict


5. Stop and remove all containers:
docker compose down


This setup demonstrates how the FastAPI application and MLflow service can be deployed and orchestrated using Docker Compose.

## AI Tools Used
ChatGPT was used to assist with:
- Debugging FastAPI and Docker issues
- Structuring API endpoints
- Generating example request formats

All code was reviewed and understood before submission.